In [3]:
from typing import Optional
from pydantic import BaseModel, Field, EmailStr, HttpUrl, SecretStr
from pydantic import field_validator, model_validator
from uuid import UUID, uuid4


class UserProfile(BaseModel):
    id: UUID = Field(default_factory=uuid4)
    email: EmailStr
    name: str
    password: SecretStr
    website: Optional[HttpUrl] = None
    bio: Optional[str] = None

    @field_validator("name")
    @classmethod
    def normalize_name(cls, v: str) -> str:
        if not v:
            return v
        normalized = " ".join(word.strip().capitalize() for word in v.split())
        return normalized

    @field_validator("password")
    @classmethod
    def password_strength(cls, v: SecretStr) -> SecretStr:
        if len(v.get_secret_value()) < 8:
            raise ValueError("Короткий пароль")
        return v

    @model_validator(mode="after")
    def check_domains(self):
        if self.website and self.email:
            email_domain = self.email.split('@')[-1].lower()
            website_domain = self.website.host.lower()

            if website_domain.startswith('www.'):
                website_domain = website_domain[4:]

            if email_domain == website_domain:
                raise ValueError("Сайт не может совпадать с почтой")
        return self

user = UserProfile(
    email="test@gmail.com",
    name="NAME",
    password="123456789",
    website="https://website.com",
    bio="PeopleHuman"
    )
print(user)

id=UUID('42e3360d-aa0e-4e9e-ac79-7591ab597075') email='test@gmail.com' name='Name' password=SecretStr('**********') website=HttpUrl('https://website.com/') bio='PeopleHuman'


In [5]:
from pydantic import validate_call, Field
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID
from typing import Dict
from uuid import uuid4 
from pydantic import validate_call
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID


@validate_call
def place_order(
    user_id: UUID,
    sku: str,
    quantity: int,
    price: Decimal
):



    if not sku.isupper() or not sku.isalnum():
        raise ValueError("должно содержать только большие буквы")

    if len(sku) < 3 or len(sku) > 12:
        raise ValueError("должно быть от 3 до 12 символов")

    if quantity <= 0:
        raise ValueError("количество должно быть больше 0")

    if price < 0:
        raise ValueError("цена не может быть отрицательной")


    rounded_price = price.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)

    amount = quantity * rounded_price

    return {
        "user_id": user_id,
        "sku": sku,
        "quantity": quantity,
        "price": rounded_price,
        "amount": amount
    }
test_id = uuid4()

result = place_order(
    user_id=test_id,
    sku="ABC123",
    quantity=2,
    price=Decimal("19.99")
)

print(result)

{'user_id': UUID('16bc87e4-c888-4911-8b17-2768073b2e65'), 'sku': 'ABC123', 'quantity': 2, 'price': Decimal('19.99'), 'amount': Decimal('39.98')}


In [9]:
from pydantic import BaseModel, EmailStr, field_validator, model_validator, Field
from typing import List
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID, uuid4
from datetime import datetime
from enum import Enum
import re

SKU_RE = re.compile(r"^[A-Z0-9]{3,12}$")


class OrderStatus(str, Enum):
    new = "new"  
    paid = "paid"  
    delivered = "delivered"  
    canceled = "canceled"  


class OrderItem(BaseModel):
    sku: str  
    qty: int  
    unit_price: Decimal  

    @field_validator("sku")
    @classmethod
    def sku_format(cls, value: str) -> str:
        if not SKU_RE.match(value):
            raise ValueError("Только заглавные буквы длиной 3-12")
        return value

    @field_validator("qty")
    @classmethod
    def qty_positive(cls, value: int) -> int:
        if value <= 0:
            raise ValueError("Должно быть больше 0")
        return value

    @field_validator("unit_price")
    @classmethod
    def price_non_negative(cls, value: Decimal) -> Decimal:
        if value < 0:
            raise ValueError("Не может быть отрицательной")
        return value.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)


class Order(BaseModel):
    id: UUID = Field(default_factory=uuid4)  
    user_email: EmailStr  
    items: List[OrderItem]  
    status: OrderStatus = OrderStatus.new  
    created_at: datetime = Field(default_factory=datetime.utcnow)  

    @property
    def total(self) -> Decimal:
        total_amount = Decimal('0')
        for item in self.items:
            total_amount += item.qty * item.unit_price
        return total_amount.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)

    @model_validator(mode="after")
    def check_business_rules(self):
        if not self.items:
            raise ValueError("Не может быть пустым")

        if self.status in [OrderStatus.paid, OrderStatus.delivered] and self.total == 0:
            raise ValueError(f"Нельзя сделать '{self.status}' сумму равной 0")

        return self


order1 = Order(
    user_email="test@example.com",
    items=[
        OrderItem(sku="ABC123", qty=1, unit_price=Decimal("999.99")),
        OrderItem(sku="DEF456", qty=2, unit_price=Decimal("29.50"))
    ]
)

print("Заказ создан")
print(f"ID: {order1.id}")
print(f"Email: {order1.user_email}")
print(f"Статус: {order1.status}")
print(f"Количество товаров: {len(order1.items)}")
print(f"Сумма: ${order1.total}")
print(f"Дата создания: {order1.created_at}")

Заказ создан
ID: 70d02a97-0014-4a7f-8de3-abad51fec7e5
Email: test@example.com
Статус: OrderStatus.new
Количество товаров: 2
Сумма: $1058.99
Дата создания: 2025-11-18 19:42:35.817051


In [2]:
import os
from pydantic_settings import BaseSettings
from pydantic import ConfigDict, SecretStr, HttpUrl, field_validator
from typing import Optional


class APISettings(BaseSettings):

    base_url: HttpUrl
    token: SecretStr


    timeout_sec: int = 5
    retries: int = 2

    @field_validator("timeout_sec", "retries")
    @classmethod
    def check_ranges(cls, value: int, info):
        field_name = info.field_name

        if field_name == "timeout_sec":
            if value < 1 or value > 60:
                raise ValueError("Таймаут 1-60 секунд")

        elif field_name == "retries":
            if value < 0 or value > 10:
                raise ValueError("Количество повторов 0-10")

        return value

    model_config = ConfigDict(
        env_prefix="API_",
        env_file=".env",
        extra="ignore"
    )



settings1 = APISettings(
    base_url="https://api.com",
    token=SecretStr("my-secret-token-123"),
    timeout_sec=10,
    retries=3
)

print("Test")
print(f"Base URL: {settings1.base_url}")
print(f"Token: {settings1.token}")
print(f"Timeout: {settings1.timeout_sec} сек")
print(f"Retries: {settings1.retries}")

Test
Base URL: https://api.com/
Token: **********
Timeout: 10 сек
Retries: 3


In [ ]:
from typing import Optional
from sqlalchemy import Column, String, Boolean
from sqlalchemy.orm import declarative_base
from uuid import uuid4, UUID
from pydantic import BaseModel, EmailStr, ConfigDict


Base = declarative_base()

class SAUser(Base):
    __tablename__ = "users"

    # Поля таблицы
    id = Column(String, primary_key=True, default=lambda: str(uuid4()))
    email = Column(String, nullable=False)
    is_active = Column(Boolean, default=True)

    def __init__(self, email: str, is_active: bool = True):
        self.id = str(uuid4())
        self.email = email
        self.is_active = is_active


class UserOut(BaseModel):

    id: UUID
    email: EmailStr
    is_active: bool

    model_config = ConfigDict(
        from_attributes=True
    )


sa_user = SAUser(
    email="test@mail.com",
    is_active=True
)

print(f"   ID: {sa_user.id} (тип: {type(sa_user.id)})")
print(f"   Email: {sa_user.email}")
print(f"   Active: {sa_user.is_active}")

In [3]:
from pydantic import BaseModel, EmailStr, field_validator, model_validator, Field
from typing import List, Tuple
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID, uuid4
from datetime import datetime
from enum import Enum
import re

SKU_RE = re.compile(r"^[A-Z0-9]{3,12}$")


class OrderStatus(str, Enum):
    new = "new"
    paid = "paid"
    delivered = "delivered"
    canceled = "canceled"


class OrderItem(BaseModel):
    sku: str
    qty: int
    unit_price: Decimal

    @field_validator("sku")
    @classmethod
    def sku_format(cls, value: str) -> str:
        if not SKU_RE.match(value):
            raise ValueError("Только заглавные буквы длиной 3-12")
        return value

    @field_validator("qty")
    @classmethod
    def qty_positive(cls, value: int) -> int:
        if value <= 0:
            raise ValueError("Должно быть больше 0")
        return value

    @field_validator("unit_price")
    @classmethod
    def price_non_negative(cls, value: Decimal) -> Decimal:
        if value < 0:
            raise ValueError("Не может быть отрицательной")
        return value.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)


class Order(BaseModel):
    id: UUID = Field(default_factory=uuid4)
    user_email: EmailStr
    items: List[OrderItem]
    status: OrderStatus = OrderStatus.new
    created_at: datetime = Field(default_factory=datetime.utcnow)

    @property
    def total(self) -> Decimal:
        total_amount = Decimal('0')
        for item in self.items:
            total_amount += item.qty * item.unit_price
        return total_amount.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)

    @model_validator(mode="after")
    def check_business_rules(self):
        if not self.items:
            raise ValueError("Не может быть пустым")

        if self.status in [OrderStatus.paid, OrderStatus.delivered] and self.total == 0:
            raise ValueError(f"Нельзя сделать '{self.status}' сумму равной 0")

        return self



ORDER_SCHEMA = Order.model_json_schema()


def safe_create_order(data: dict) -> Tuple[bool, str]:

    try:
        order = Order(**data)

        return True, f"total={order.total}"

    except ValidationError as e:
        error_message = str(e.errors()[0]['msg'])
        return False, error_message

    except Exception as e:
        return False, str(e)


print(f"Заголовок: {ORDER_SCHEMA.get('title', 'N/A')}")
print(f"Количество свойств: {len(ORDER_SCHEMA.get('properties', {}))}")

Заголовок: Order
Количество свойств: 5
